# gl-yolo — sweep complet (MVA → DOTA → VisDrone)

Entraîne les 8 architectures sur les 3 datasets (24 runs), 200 epochs, imgsz=1280.

**Avant de lancer** : Runtime → Change runtime type → GPU (A100 si Colab Pro+ le propose).

**Ce run va prendre plusieurs jours.** Le repo est cloné *dans Drive* (pas sur le disque local éphémère) pour que `runs/` et `logs/` survivent aux déconnexions de session : à chaque reconnexion, ré-exécutez les cellules dans l'ordre — `run_gpu.py` saute automatiquement les modèles déjà terminés et reprend le modèle interrompu depuis son dernier checkpoint au lieu de le refaire depuis zéro.

## 1. Vérifier le GPU attribué

In [ ]:
!nvidia-smi
import torch
print("CUDA disponible:", torch.cuda.is_available())

## 2. Monter Google Drive (persistance entre sessions)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Cloner (1re fois) ou mettre à jour le repo sur Drive

In [ ]:
import os

REPO_DIR = "/content/drive/MyDrive/gl-yolo"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/osiastossou/gl-yolo.git "{REPO_DIR}"
else:
    print("Repo déjà présent sur Drive, mise à jour...")
    !git -C "{REPO_DIR}" pull

%cd {REPO_DIR}

## 4. Installer les dépendances

In [ ]:
!pip install -q -e .
!pip install -q gdown

## 5. (Optionnel) vérifier l'état d'un sweep précédent

Utile après une reconnexion, pour voir ce qui est déjà fini avant de relancer.

In [ ]:
!find runs -name best.pt 2>/dev/null
!find runs -name last.pt 2>/dev/null

## 6. Lancer le sweep complet

MVA (bird, single-class) en premier, puis DOTA, puis VisDrone — 8 architectures chacun.

Activez l'**exécution en arrière-plan** (icône dans la barre d'outils Colab, fonctionnalité Pro+) avant de lancer cette cellule pour ne pas perdre le run en fermant l'onglet.

In [ ]:
!python run_gpu.py -epochs 200 -imgsz 1280

## En cas de déconnexion

Reconnectez-vous, ré-exécutez les cellules **2 à 6** dans l'ordre (Drive → repo → deps → sweep). `run_gpu.py` saute les modèles déjà terminés (`best.pt` présent) et reprend automatiquement celui qui était en cours (`last.pt` sans `best.pt`) depuis son dernier checkpoint.